# Scoring by Player Origin - both previous team and hometown

## Dependencies and Setup

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import regex as re

import numpy as np
import requests
from bs4 import BeautifulSoup
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import matplotlib.font_manager as fm
from matplotlib.font_manager import FontProperties
from matplotlib.offsetbox import OffsetImage
from matplotlib.ticker import PercentFormatter
from matplotlib.ticker import ScalarFormatter
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from PIL import Image

# ======= BASE PATHS =======
try:
    # Works when running as a script
    base_dir = Path(__file__).resolve().parent
except NameError:
    # Fallback for notebooks or interactive mode
    base_dir = Path.cwd()

project_root = base_dir.parent.parent


# ======= DATA FOLDERS =======
temp_folder = project_root / "TEMP"
data_folder = project_root / "data"
roster_folder = data_folder / "player_info"
school_info_folder = data_folder / "school_info"

# ======= IMAGE FOLDERS =======
img_folder = project_root / "images"
logo_folder = img_folder / "logos"
background_folder = img_folder / "background"
plot_folder = project_root / "TEMP" / "IMAGE" / "scoring_origins"

# ======= IMPORT CONFIG =======
import config  # now you can import config.py

# ======= LOAD DATA =======
roster_file = roster_folder / "roster_10_30_25.csv"
roster_df = pd.read_csv(roster_file)
roster_df["Current Team"] = roster_df["Current Team"].replace("RPI", "Rensselaer")

print(roster_df.columns)

school_info_file = school_info_folder / "arena_school_info.csv"
school_info_df = pd.read_csv(school_info_file)

# Check the Config import
# print((config_folder / "config.py").read_text())

Index(['Current Team', 'Last_Name', 'First_Name', 'No', 'Position', 'Yr', 'Ht',
       'Wt', 'DOB', 'Hometown', 'Height_Inches', 'Draft_Year', 'NHL_Team',
       'D_Round', 'Last Team', 'League', 'City', 'State_Province', 'Country'],
      dtype='object')


## Transform the Roster data for easy Merge

In [2]:
# Examine the roster data
# roster_df.head()
# roster_df.info()
# print(roster_df.columns)
# school_info_df.head()

In [3]:
## # Combine First and last name in roster_df to match player_ytd_df

# Clean white space from names
roster_df["First_Name"] = roster_df["First_Name"].str.strip()
roster_df["Last_Name"] = roster_df["Last_Name"].str.strip()
roster_df["Clean_Player"] = roster_df["First_Name"] + " " + roster_df["Last_Name"]
# Strip any leading/trailing whitespace
roster_df["Clean_Player"] = roster_df["Clean_Player"].str.strip()
# Rename Current Team to match player_ytd_df
roster_df = roster_df.rename(columns={"Current Team": "Team"})
# Reorder columns for easier viewing
#Order of columns
# ["No","Team","Clean_Player", "First_Name","Last_Name", 'Position', 'Yr', 'Ht', 'Wt', 'DOB', 'Hometown', 'Height_Inches', 'Draft_Year', 'NHL_Team', 'D_Round', 'Last Team', 'League', 'City', 'State_Province', 'Country']
roster_df = roster_df[["No","Team","Clean_Player", "First_Name","Last_Name", 'Position', 'Yr', 'Ht', 'Wt', 'DOB', 'Hometown', 'Height_Inches', 'Draft_Year', 'NHL_Team', 'D_Round', 'Last Team', 'League', 'City', 'State_Province', 'Country']]    




roster_df.head()

## State_Province Value Counts
check_states = roster_df["State_Province"].value_counts(dropna=False)
check_states


State_Province
Minnesota           217
Ontario             211
British Columbia    124
Alberta             116
New York             98
                   ... 
Delaware              1
Germany               1
JPN                   1
Utah                  1
Okla.                 1
Name: count, Length: 73, dtype: int64

In [4]:
### Roster Modifications to make up for incomplete data ###

### Replace bad values in State_Province column
state_province_corrections = {"Okla.": "Oklahoma", "D.C": "District of Columbia"
}
# Apply state/province corrections
roster_df["State_Province"] = roster_df["State_Province"].replace(state_province_corrections)

### Replace country abbreviations with full country names
country_corrections = {"CYM": "Cayman Islands", "JPN": "Japan", "RUS": "Russia"
}
# Apply country corrections
roster_df["Country"] = roster_df["Country"].replace(country_corrections)

########## HOTFIX FOR BAD LAST_TEAM / LEAGUE DATA ##########
# Correct Adam Valentini - Listed as Canada U18 but that was just a tourney. he actually played in the USHL
roster_df.loc[roster_df["Clean_Player"] == "Adam Valentini", "Last Team"] = "Chicago Steel"
roster_df.loc[roster_df["Clean_Player"] == "Adam Valentini", "League"] = "USHL"

# Correct Tanner Hartman - Listed as NCAA D3 which is screwing up the classification - change that to DIII
roster_df.loc[roster_df["Clean_Player"] == "Tanner Hartman", "League"] = "DIII"

# Correct Justin Solovey - Team Muskegon Lumberjacks, League USHL
roster_df.loc[roster_df["Clean_Player"] == "Justin Solovey", "Last Team"] = "Muskegon Lumberjacks"
roster_df.loc[roster_df["Clean_Player"] == "Justin Solovey", "League"] = "USHL"



# Create List or Dictionary of known players with bad/missing data and their correct last team/league
# Then apply those corrections to the roster_df before classification
corrections = {
    "Anthony Galante": "NAHL",
    "Charles Banquier": "BCHL",
    "Devon Carlstrom": "NAHL",
    "Dominick Campione": "AJHL",
    "Frank Ireland": "NCDC",
    "Graham Harris": "DIII",
    "Johnny Druskinis": "NCAA D1",
    "Jonathan Castagna": "PREP",
    "Joseph Grainda": "NAHL",
    "Kasper Magnussen": "Europe",
    "Maxon Vig": "USHL",
    "Mick Frechette": "PREP",
    "Nicholas Chin-DeGraves": "BCHL",
    "Nick Bernardo": "PREP",
    "Tristan Sarsland": "PREP",
    "Wilson Bjorck": "Europe",
    
}

## Apply corrections to roster_df
for player, league in corrections.items():
    roster_df.loc[roster_df["Clean_Player"] == player, "League"] = league
    







### Connect to database

In [5]:
## Connect to database using the recent_clean_db path from config.py
import sqlite3

#### CONFIG FILE NOTE WORKING AS EXPECTED - MANUAL FIX ####
data_folder = ('../../data/db/')
# filename = '2025_Feb_13_CLEAN.db'
filename = 'Season_YTD.db'
recent_clean_db = data_folder + filename
########### END MANUAL FIX ###########

conn = sqlite3.connect(recent_clean_db)
cursor = conn.cursor()
print("Connected to database:", config.recent_clean_db)


Connected to database: ../../data/db/Season_YTD.db


### Extract and merge the year to date stats
- Issue - the way the player stats ytd table is created it gives credit for games played to everyone, even if they didn't appear in a game
- ultimately I should change the scraping and aggrigation code so a player only gets credit for a game if TOI_sec is > 0

In [6]:
#### Extract and merge the year to date stats ####
player_ytd_query = """
SELECT * FROM player_stats_ytd
"""

player_ytd_df = pd.read_sql_query(player_ytd_query, conn)
# Replace RPI with Rensselaer to match roster_df
player_ytd_df["Team"] = player_ytd_df["Team"].replace("RPI", "Rensselaer")

## print length of DataFrame and columns
print("Player Stats YTD DataFrame shape:", player_ytd_df.shape)
print(player_ytd_df.columns)
# Drop any rows with TOTAL in the Clean_Player column
player_ytd_df = player_ytd_df[~player_ytd_df['Clean_Player'].str.contains('TOTAL', na=False)]

# Check length of DataFrame and columns after drop

print("Player Stats YTD DataFrame shape:", player_ytd_df.shape)
print(player_ytd_df.columns)

# Close the database connection
conn.close()



Player Stats YTD DataFrame shape: (1599, 14)
Index(['Clean_Player', 'Team', 'G', 'A', 'Pts', 'PlusMinus', 'Shots',
       'TOI_sec', 'PIM', 'FOW', 'FOL', 'Games_Played', 'FO%', 'TOI'],
      dtype='object')
Player Stats YTD DataFrame shape: (1599, 14)
Index(['Clean_Player', 'Team', 'G', 'A', 'Pts', 'PlusMinus', 'Shots',
       'TOI_sec', 'PIM', 'FOW', 'FOL', 'Games_Played', 'FO%', 'TOI'],
      dtype='object')


In [7]:
# Make sure name and team columns are stripped of punctuation, strange characters, and whitespace
player_ytd_df["Clean_Player"] = player_ytd_df["Clean_Player"].str.strip()
player_ytd_df["Team"] = player_ytd_df["Team"].str.strip()
roster_df["Clean_Player"] = roster_df["Clean_Player"].str.strip()
roster_df["Team"] = roster_df["Team"].str.strip()
# Remove any hyphens, periods, ect from team names to match
player_ytd_df["Team"] = player_ytd_df["Team"].str.replace(r'[^\w\s]', ' ', regex=True)
roster_df["Team"] = roster_df["Team"].str.replace(r'[^\w\s]', ' ', regex=True)
# QUICK FIX - Standardize team names with double spaces
# If team name column has double spaces, replace with single space
player_ytd_df["Team"] = player_ytd_df["Team"].str.replace('  ', ' ', regex=False)
roster_df["Team"] = roster_df["Team"].str.replace('  ', ' ', regex=False)

## Merge the two DataFrames on Clean_Player and Team
merged_df = pd.merge(
    player_ytd_df,
    roster_df,
    left_on=["Clean_Player", "Team"],
    right_on=["Clean_Player", "Team"],
    how="left"
)

# Shape and info of merged DataFrame
print("Merged DataFrame shape:", merged_df.shape)
# print(merged_df.columns)
# print(merged_df.head())
merged_df.info()



Merged DataFrame shape: (1599, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 32 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Clean_Player    1599 non-null   object 
 1   Team            1599 non-null   object 
 2   G               1599 non-null   int64  
 3   A               1599 non-null   int64  
 4   Pts             1599 non-null   int64  
 5   PlusMinus       1599 non-null   int64  
 6   Shots           1599 non-null   int64  
 7   TOI_sec         1599 non-null   int64  
 8   PIM             1599 non-null   int64  
 9   FOW             1599 non-null   float64
 10  FOL             1599 non-null   float64
 11  Games_Played    1599 non-null   int64  
 12  FO%             1599 non-null   float64
 13  TOI             1599 non-null   object 
 14  No              1598 non-null   float64
 15  First_Name      1598 non-null   object 
 16  Last_Name       1598 non-null   object 
 17

### NEED TO DEAL WITH THIS WEIRD ONE EDGE CASE

In [8]:
## Export merged DataFrame to CSV for examination
# output_file = "../../TEMP/merged_player_stats_roster_test_1.csv"

## Show me a random selection of ten rows that didn't match for merge
## ie no current team, last name, ect
missing_team_df = merged_df[merged_df["First_Name"].isnull()]

missing_team_df


,Clean_Player,Team,G,A,Pts,PlusMinus,Shots,TOI_sec,PIM,FOW,...,Hometown,Height_Inches,Draft_Year,NHL_Team,D_Round,Last Team,League,City,State_Province,Country
1081,Maxon Vig,Bemidji State,1,1,2,8,13,13964,10,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Cleaning and Validation

### Filter out players with no TOI and goalies
- Remove players that haven't appeared in a game at all this year (TOI_sec = 0)
    -can't depend on YTD stats games played until we fix the scraping code
- Filter out goaltenders as non skaters - looking at offensive production so goalies are irrelivant (despite the assists they may get from time to time)



In [9]:
## Print Value COunt of Position column
# print(merged_df['Position'].value_counts())

In [10]:
### Print dataframe stats for to keep track of filtering steps
original_count = merged_df.shape[0]
print("Merged DataFrame shape before filtering:", merged_df.shape)

### Filter out players with no TOI and goalies
# Remove players that haven't appeared in a game at all this year (TOI_sec = 0)
# Remove any Rows where TOI_sec is 0 or NaN - These are goalies or players with no time on ice
merged_df = merged_df[(merged_df["TOI_sec"] > 0) & (~merged_df["TOI_sec"].isna())]
first_step_count = merged_df.shape[0]

## Check the shape after filtering
print("Merged DataFrame shape after filtering TOI_sec > 0:", merged_df.shape)
# Number of players removed
print("Players removed after filtering TOI_sec > 0:", original_count - first_step_count)

### DO NOT NEED TO FILTER FOR GOALIES BECAUSE PLAYER_YTD_STATS TABLE ONLY HAS TOI FOR SKATERS
# Filter out goaltenders as non skaters - looking at offensive production so goalies are irrelivant (despite the assists they may get from time to time)
# Strip any whitespace from Position column
# merged_df["Position"] = merged_df["Position"].str.strip()
# merged_df = merged_df[merged_df["Position"] != "Goaltenders"]
# no_goalie_count = merged_df.shape[0]
## Check the shape after filtering
# print("Merged DataFrame shape after filtering out Goalies:", merged_df.shape)
# # Number of players removed
# print("Players removed by filtering out Goalies:", first_step_count - no_goalie_count)

Merged DataFrame shape before filtering: (1599, 32)
Merged DataFrame shape after filtering TOI_sec > 0: (1464, 32)
Players removed after filtering TOI_sec > 0: 135


In [11]:
# merged_df.columns

## Quick Explore of Origin Data

In [12]:
## Value count of 'League' column
# print(merged_df['League'].value_counts())

# ### Country value counts
# print(merged_df['Country'].value_counts())

# # Check for any players that have null or 0 TOI but have Games Played > 0
# null_toi_df = merged_df[(merged_df["TOI_sec"].isnull()) | (merged_df["TOI_sec"] == 0) & (merged_df["Games_Played"] > 0)]
# null_toi_df




In [13]:
# Correct Maxon Vig - Team Cedar Rapids RoughRiders, League USHL
## Couldn't do it before because he was not in original roster_df but now he is in merged_df
merged_df.loc[merged_df["Clean_Player"] == "Maxon Vig", "Last Team"] = "Cedar Rapids RoughRiders"
merged_df.loc[merged_df["Clean_Player"] == "Maxon Vig", "League"] = "USHL"

### Clean and Classify Previous Team / League columns
- put into the same bins as used on the Team Construction Visual
- Copied code from there with a few changes made based on feedback / missed classifications 

In [14]:
### Reusing League and Team Classification function and libraries from team_construction_visual_workbook import classify_previous_team, classify_previous_league

# ----------------------------
# 1) Rename merged_df to df for easier to fit in with existing code
# -----------------------------
df = merged_df.copy()

# -----------------------------
# 2) Classification helpers
# -----------------------------
def _norm_set(strings):
    return { _norm(s) for s in strings }

def _norm(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().upper()
    s = re.sub(r"[.\u2010-\u2015\-–—]+", " ", s)  # unify hyphen-like chars to space
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _norm_set(strings):
    return { _norm(s) for s in strings }

NTDP_TEAM_HINTS_RAW = (
    "USA U 18", "US U 18", "USA U18", "US U18",
    "USA U 17", "US U 17", "USA U17", "US U17",
    "NTDP", "USNTDP", "US NATIONAL TEAM", "US DEV PROGRAM", "US DEVELOPMENT PROGRAM"
)
NTDP_LEAGUE_HINTS_RAW = ("NTDP",)

D1_CONFS_RAW = {"ECAC","CCHA","NCHC","HEA","B10","AHA","INDEPENDENTS","D I IND","NCAA"}

CJHL_LEAGUES_RAW = {"BCHL","AJHL","SJHL","OJHL", "MJHL", "CCHL","MHL"}

US_TIER1_RAW = {"USHL"}
US_TIER2_RAW = {"NAHL","NCDC"}
US_OTHER_RAW  = {"USPHL","NA3HL", "PREP", "PHC", "USHS", "CISAA", "DIII", "D-III"}

CHL_RAW = {"OHL","WHL","QMJHL", "MJAHL"}

EURO_HINTS_RAW = {
    "J20 NATIONELL","J18 REGION","U20 SM SARJA","U18","U20","SM SARJA",
    "SHL","ICEHL","ALPSHL","LIIGA","MHL RUSSIA","KHL, ICEHL","KHL","DEL", "EC-KAC",
}

EURO_TEAM_HINTS_RAW = {
    "KalPa U20", "Frölunda HC", "Malmo", "Jokerit U20", "U20 SM Sarja-Pelicans",
    "Tappara J20", "Djurgårdens IF", "Leksands IF",
}

RUSSIAN_MHL_TEAM_HINTS_RAW = (
    "KRASNAYA","LOKO","MOSKVA","MOSCOW","ST PETERSBURG","SKA","LOKOMOTIV",
    "DMITROV","CHELYABINSK","OMSK","NOVOSIBIRSK","MAGNITOGORSK","NIZHNY","YAROSLAVL",
)

# Normalize them
NTDP_TEAM_HINTS       = _norm_set(NTDP_TEAM_HINTS_RAW)
NTDP_LEAGUE_HINTS     = _norm_set(NTDP_LEAGUE_HINTS_RAW)
D1_CONFS              = _norm_set(D1_CONFS_RAW)
CJHL_LEAGUES          = _norm_set(CJHL_LEAGUES_RAW)
US_TIER1              = _norm_set(US_TIER1_RAW)
US_TIER2              = _norm_set(US_TIER2_RAW)
US_OTHER              = _norm_set(US_OTHER_RAW)
CHL                   = _norm_set(CHL_RAW)
EURO_HINTS            = _norm_set(EURO_HINTS_RAW)
EURO_TEAM_HINTS       = _norm_set(EURO_TEAM_HINTS_RAW)
RUSSIAN_MHL_TEAM_HINTS = _norm_set(RUSSIAN_MHL_TEAM_HINTS_RAW)

PRO_HINTS = ("AHL","ECHL")

BIN_ORDER = [
    "NTDP",
    "USHL (non‑NTDP)",
    "NAHL/NCDC",
    "US (DIII/Prep/Other)",
    "CHL (Major Junior)",
    "CJHL (Canadian Jr A)",
    "U SPORTS",
    "Europe",
    "NCAA D1 Transfers",
    "Pro (AHL/ECHL/Other)",
    "Other/Various/Unknown"
    
]

COLOR_MAP = {
    "NTDP": "#0057B8",
    "USHL (non‑NTDP)": "#1E90FF",
    "NAHL/NCDC": "#63B8FF",
    "US (DIII/Prep/Other)": "#B0E2FF",
    "CHL (Major Junior)": "#B22222",
    "CJHL (Canadian Jr A)": "#FF7F7F",
    "U SPORTS": "#FADBD8",
    "Europe": "#F0E130",
    "NCAA D1 Transfers": "#696969",
    "Pro (AHL/ECHL/Other)": "#000000",
    "Other/Various/Unknown": "#A9A9A9"
}

def classify_prev_bin(last_team: str, league: str) -> str:
    t = _norm(last_team)
    l = _norm(league)

    # NTDP carve-out first
    if any(h in t for h in NTDP_TEAM_HINTS) or any(h == l for h in NTDP_LEAGUE_HINTS):
        return "NTDP"

    # D3/Prep -> Other
    if l in US_OTHER:
        return "US (DIII/Prep/Other)"

    # NCAA D1 transfers
    if (l in D1_CONFS) or ("NCAA" in l and l != ""):
        return "NCAA D1 Transfers"

    # U SPORTS
    if l in {"USPORTS", "U SPORTS"}:
        return "U SPORTS"

    # CHL
    if l in CHL:
        return "CHL (Major Junior)"

    # MHL ambiguity
    if l == "MHL":
        if any(k in t for k in RUSSIAN_MHL_TEAM_HINTS):
            return "Europe"
        else:
            return "CJHL (Canadian Jr A)"

    # CJHL
    if l in CJHL_LEAGUES:
        return "CJHL (Canadian Jr A)"



    # US juniors
    if l in US_TIER1:
        return "USHL (non‑NTDP)"
    if l in US_TIER2:
        return "NAHL/NCDC"
    
    # Pro leagues
    if any(h in l for h in PRO_HINTS):
        return "Pro (AHL/ECHL/Other)"

    # Europe consolidated
    if l in EURO_HINTS:
        return "Europe"
    if t in EURO_TEAM_HINTS:
        return "Europe"




    # Fallbacks/Unknown
    # if l == "" or pd.isna(league):
    #     return "Other/Various/Unknown"
    
    if "U SPORTS" in t:
        return "U SPORTS"
    if "IF Sundsvall" in t:  # edge case
        return "Europe"


    return "Other/Various/Unknown"


# Apply classification to the raw roster
df["Prev_League_Bin"] = df.apply(lambda r: classify_prev_bin(r.get("Last Team", np.nan),
                                                             r.get("League", np.nan)), axis=1)
df["Prev_League_Bin"] = pd.Categorical(df["Prev_League_Bin"], categories=BIN_ORDER, ordered=True)

# Save classified to temp_folder
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# out_file = os.path.join(temp_folder, f"roster_classified_{timestamp}.csv")
# df.to_csv(out_file, index=False)


# show quick counts
overall_counts = (df["Prev_League_Bin"]
                  .value_counts(dropna=False)
                  .reindex(BIN_ORDER)
                  .fillna(0).astype(int))

print("Overall Counts by Prev_League_Bin")
print(overall_counts.reset_index().rename(columns={"index":"Bin","Prev_League_Bin":"Count"}))

Overall Counts by Prev_League_Bin
                    Count  count
0                    NTDP     41
1         USHL (non‑NTDP)    440
2               NAHL/NCDC    200
3    US (DIII/Prep/Other)     11
4      CHL (Major Junior)    148
5    CJHL (Canadian Jr A)    318
6                U SPORTS     34
7                  Europe     24
8       NCAA D1 Transfers    242
9    Pro (AHL/ECHL/Other)      5
10  Other/Various/Unknown      1


In [15]:
## Show The Other/Various/Unknown players
other_unknown_df = df[df["Prev_League_Bin"] == "Other/Various/Unknown"]
other_unknown_df = other_unknown_df[["Clean_Player", "Team", "Last Team", "League", "Prev_League_Bin" ]]
# other_unknown_df

### Generate table of the number of players on each team

In [16]:
### Create a table of the number of players on each team from the various leagues
team_league_counts = (df
                      .groupby(["Team", "Prev_League_Bin"])
                      .size()
                      .unstack(fill_value=0)
                      .reindex(columns=BIN_ORDER, fill_value=0)
                     )  

# Reorder so CHL is first column after Team
cols = team_league_counts.columns.tolist()
cols = cols[-6:] + cols[:-1]  # Move last column to first position
team_league_counts = team_league_counts[cols]



# team_league_counts                  

C:\Users\jbanc\AppData\Local\Temp\ipykernel_18616\3039452061.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["Team", "Prev_League_Bin"])


### Adgrigate Stats using the new class bins

In [17]:
### Group by Prev_League_Bin and aggregate stats (G, A, Pts, PlusMinus, Shots, TOI_sec, PIM)

agg_stats = {
    "Games_Played": "sum",
    "G": "sum",
    "A": "sum",
    "Pts": "sum",
    "PlusMinus": "sum",
    "Shots": "sum",
    "TOI_sec": "sum",
    "PIM": "sum",
}
agg_df = df.groupby("Prev_League_Bin").agg(agg_stats).reset_index()
# # creat column for count of players in each bin
# agg_df["Player_Count"] = df["Prev_League_Bin"].value_counts().reindex(agg_df["Prev_League_Bin"]).values
# # Create AVG games played per player column
# agg_df["Avg_Games Played"] = agg_df["Games_Played"] / agg_df["Player_Count"]

# # Reorder Columns to put player_count right after Prev_League_Bin
# agg_df = agg_df[["Prev_League_Bin", "Player_Count", "Games_Played", "Avg_Games Played"] + list(agg_stats.keys())]


agg_df

C:\Users\jbanc\AppData\Local\Temp\ipykernel_18616\1377335929.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg_df = df.groupby("Prev_League_Bin").agg(agg_stats).reset_index()


,Prev_League_Bin,Games_Played,G,A,Pts,PlusMinus,Shots,TOI_sec,PIM
0,NTDP,445,110,146,256,37,835,439060,349
1,USHL (non‑NTDP),4100,650,1129,1779,201,6609,3878100,2261
2,NAHL/NCDC,1540,157,301,458,-217,1971,1287739,794
3,US (DIII/Prep/Other),93,20,22,42,9,183,88779,50
4,CHL (Major Junior),1469,251,423,674,-19,2507,1354624,925
5,CJHL (Canadian Jr A),2589,341,585,926,-140,3609,2232978,1215
6,U SPORTS,327,58,101,159,-19,539,316723,251
7,Europe,196,37,56,93,21,275,170340,91
8,NCAA D1 Transfers,2472,416,691,1107,-45,4196,2428566,1440
9,Pro (AHL/ECHL/Other),53,13,20,33,5,94,62022,57


In [18]:
## Get count of players in each Prev_League_Bin
player_counts = (df["Prev_League_Bin"]
                  .value_counts(dropna=False)
                  .reindex(BIN_ORDER)
                  .fillna(0).astype(int))

# print("Player Counts by Prev_League_Bin")
# print(player_counts.reset_index().rename(columns={"index":"Bin","Prev_League_Bin":" Count"}))

In [19]:
### Divide aggregated stats by player counts to get per player averages
for stat in agg_stats.keys():
    agg_df[stat + "_per_player"] = agg_df[stat] / player_counts.values
# agg_df

## Use the Games_Played column to get per game averages for all stats
for stat in agg_stats.keys():
    agg_df[stat + "_per_game"] = agg_df[stat] / agg_df["Games_Played"]

#### Calulate per 60 minutes stats for each aggregated stat
for stat in agg_stats.keys():
    agg_df[stat + "_per_60min"] = (agg_df[stat] / agg_df["TOI_sec"]) * 3600

In [20]:
# #### Calulate per 60 minutes stats for each aggregated stat
# for stat in agg_stats.keys():
#     agg_df[stat + "_per_60min"] = (agg_df[stat] / (agg_df["TOI_sec"]) * 3600)


In [21]:
# create column for count of players in each bin
agg_df["Player_Count"] = df["Prev_League_Bin"].value_counts().reindex(agg_df["Prev_League_Bin"]).values
# Create AVG games played per player column
agg_df["Avg_Games Played"] = agg_df["Games_Played"] / agg_df["Player_Count"]

# Move Player_Count column to be right after Prev_League_Bin and Avg_Games Played
cols = agg_df.columns.tolist()
cols.insert(1, cols.pop(cols.index("Player_Count")))
cols.insert(3, cols.pop(cols.index("Avg_Games Played")))
agg_df = agg_df[cols]

In [22]:
### Examine the final aggregated DataFrame
agg_df.info()
agg_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   Prev_League_Bin          11 non-null     category
 1   Player_Count             11 non-null     int64   
 2   Games_Played             11 non-null     int64   
 3   Avg_Games Played         11 non-null     float64 
 4   G                        11 non-null     int64   
 5   A                        11 non-null     int64   
 6   Pts                      11 non-null     int64   
 7   PlusMinus                11 non-null     int64   
 8   Shots                    11 non-null     int64   
 9   TOI_sec                  11 non-null     int64   
 10  PIM                      11 non-null     int64   
 11  Games_Played_per_player  11 non-null     float64 
 12  G_per_player             11 non-null     float64 
 13  A_per_player             11 non-null     float64 
 14  Pts_per_play

,Prev_League_Bin,Player_Count,Games_Played,Avg_Games Played,G,A,Pts,PlusMinus,Shots,TOI_sec,...,TOI_sec_per_game,PIM_per_game,Games_Played_per_60min,G_per_60min,A_per_60min,Pts_per_60min,PlusMinus_per_60min,Shots_per_60min,TOI_sec_per_60min,PIM_per_60min
0,NTDP,41,445,10.853659,110,146,256,37,835,439060,...,986.651685,0.784270,3.648704,0.901927,1.197103,2.099030,0.303375,6.846445,3600.0,2.861568
1,USHL (non‑NTDP),440,4100,9.318182,650,1129,1779,201,6609,3878100,...,945.878049,0.551463,3.805987,0.603388,1.048039,1.651427,0.186586,6.135066,3600.0,2.098863
2,NAHL/NCDC,200,1540,7.700000,157,301,458,-217,1971,1287739,...,836.194156,0.515584,4.305220,0.438909,0.841475,1.280384,-0.606645,5.510123,3600.0,2.219704
3,US (DIII/Prep/Other),11,93,8.454545,20,22,42,9,183,88779,...,954.612903,0.537634,3.771162,0.811003,0.892103,1.703105,0.364951,7.420674,3600.0,2.027507
4,CHL (Major Junior),148,1469,9.925676,251,423,674,-19,2507,1354624,...,922.140231,0.629680,3.903962,0.667049,1.124150,1.791198,-0.050494,6.662513,3600.0,2.458247
5,CJHL (Canadian Jr A),318,2589,8.141509,341,585,926,-140,3609,2232978,...,862.486674,0.469293,4.173978,0.549759,0.943135,1.492894,-0.225708,5.818418,3600.0,1.958819
6,U SPORTS,34,327,9.617647,58,101,159,-19,539,316723,...,968.571865,0.767584,3.716812,0.659251,1.148006,1.807257,-0.215962,6.126489,3600.0,2.852966
7,Europe,24,196,8.166667,37,56,93,21,275,170340,...,869.081633,0.464286,4.142304,0.781965,1.183515,1.965481,0.443818,5.811906,3600.0,1.923212
8,NCAA D1 Transfers,242,2472,10.214876,416,691,1107,-45,4196,2428566,...,982.429612,0.582524,3.664385,0.616660,1.024308,1.640968,-0.066706,6.219967,3600.0,2.134593
9,Pro (AHL/ECHL/Other),5,53,10.600000,13,20,33,5,94,62022,...,1170.226415,1.075472,3.076328,0.754571,1.160878,1.915449,0.290220,5.456128,3600.0,3.308503


### Plan for Visualizations
- data transformation is looking good to this point. SHould have a few diff. interesting views

- Should I use stacked bars witht he country bins and color grtadiant within?
- Should come up a good visual for the number of games played by each bin of players

## Bin By Country

In [23]:
### Do The Aggrigation by Home Country
country_agg_stats = {
    "Games_Played": "sum",
    "G": "sum",
    "A": "sum",
    "Pts": "sum",
    "PlusMinus": "sum",
    "Shots": "sum",
    "TOI_sec": "sum",
    "PIM": "sum",
}
country_agg_df = merged_df.groupby("Country").agg(country_agg_stats).reset_index()
print(country_agg_df.head(20))

## Get count of players in each Country and calculate per player averages
country_player_counts = (df["Country"]
                    .value_counts(dropna=False)
                    .fillna(0).astype(int))
# print("Player Counts by Country")
# print(country_player_counts.reset_index().rename(columns={"index":"Country","Country":" Count"}))

           Country  Games_Played    G     A   Pts  PlusMinus  Shots  TOI_sec  \
0          Austria            28    8    14    22          5     51    27816   
1          Belarus            16    5     4     9         -2     16    11815   
2           Canada          4912  790  1343  2133       -124   7891  4512571   
3   Cayman Islands            11    0     4     4         -5     10    10907   
4          Croatia             8    1     1     2          4     18     4501   
5   Czech Republic            64   15    22    37         -5    132    67401   
6          Czechia            11    1     2     3         -8     21    10394   
7          Finland           173   31    40    71          2    247   151420   
8          Germany            11    1     3     4         -1     19    10620   
9    Great Britain             9    1     1     2          1     10     6306   
10         Hungary            10    1     2     3          3      8     9372   
11           Italy            12    0   

In [24]:
## Calculate per player averages for each aggregated stat by Country
country_player_counts = country_player_counts.reindex(country_agg_df["Country"])
for stat in country_agg_stats.keys():
    country_agg_df[stat + "_per_player"] = country_agg_df[stat] / country_player_counts.values

# Calculate per game averages for each aggregated stat by Country
for stat in country_agg_stats.keys():
    country_agg_df[stat + "_per_game_played"] = country_agg_df[stat] / country_agg_df["Games_Played"]

## Calculate rate stats per 60 minutes for each aggregated stat by Country
for stat in country_agg_stats.keys():
    country_agg_df[stat + "_per_60min"] = (country_agg_df[stat] / country_agg_df["TOI_sec"]) * 3600

country_agg_df.info()
country_agg_df.head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 33 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Country                       23 non-null     object 
 1   Games_Played                  23 non-null     int64  
 2   G                             23 non-null     int64  
 3   A                             23 non-null     int64  
 4   Pts                           23 non-null     int64  
 5   PlusMinus                     23 non-null     int64  
 6   Shots                         23 non-null     int64  
 7   TOI_sec                       23 non-null     int64  
 8   PIM                           23 non-null     int64  
 9   Games_Played_per_player       23 non-null     float64
 10  G_per_player                  23 non-null     float64
 11  A_per_player                  23 non-null     float64
 12  Pts_per_player                23 non-null     float64
 13  PlusMin

,Country,Games_Played,G,A,Pts,PlusMinus,Shots,TOI_sec,PIM,Games_Played_per_player,...,TOI_sec_per_game_played,PIM_per_game_played,Games_Played_per_60min,G_per_60min,A_per_60min,Pts_per_60min,PlusMinus_per_60min,Shots_per_60min,TOI_sec_per_60min,PIM_per_60min
0,Austria,28,8,14,22,5,51,27816,8,9.333333,...,993.428571,0.285714,3.623814,1.035375,1.811907,2.847282,0.647110,6.600518,3600.0,1.035375
1,Belarus,16,5,4,9,-2,16,11815,4,8.000000,...,738.437500,0.250000,4.875159,1.523487,1.218790,2.742277,-0.609395,4.875159,3600.0,1.218790
2,Canada,4912,790,1343,2133,-124,7891,4512571,2876,9.147114,...,918.683021,0.585505,3.918653,0.630239,1.071407,1.701646,-0.098924,6.295214,3600.0,2.294390
3,Cayman Islands,11,0,4,4,-5,10,10907,4,5.500000,...,991.545455,0.363636,3.630696,0.000000,1.320253,1.320253,-1.650316,3.300633,3600.0,1.320253
4,Croatia,8,1,1,2,4,18,4501,2,8.000000,...,562.625000,0.250000,6.398578,0.799822,0.799822,1.599645,3.199289,14.396801,3600.0,1.599645
5,Czech Republic,64,15,22,37,-5,132,67401,36,10.666667,...,1053.140625,0.562500,3.418347,0.801175,1.175057,1.976232,-0.267058,7.050340,3600.0,1.922820
6,Czechia,11,1,2,3,-8,21,10394,2,11.000000,...,944.909091,0.181818,3.809890,0.346354,0.692707,1.039061,-2.770829,7.273427,3600.0,0.692707
7,Finland,173,31,40,71,2,247,151420,75,8.238095,...,875.260116,0.433526,4.113063,0.737023,0.950997,1.688020,0.047550,5.872408,3600.0,1.783120
8,Germany,11,1,3,4,-1,19,10620,4,11.000000,...,965.454545,0.363636,3.728814,0.338983,1.016949,1.355932,-0.338983,6.440678,3600.0,1.355932
9,Great Britain,9,1,1,2,1,10,6306,8,9.000000,...,700.666667,0.888889,5.137964,0.570885,0.570885,1.141770,0.570885,5.708849,3600.0,4.567079


## Bin By State / Province

In [25]:
## Examine the State_Province column value counts
state_prov_counts = merged_df['State_Province'].value_counts()
print("State/Province Value Counts")
print(state_prov_counts)

State/Province Value Counts
State_Province
Minnesota           184
Ontario             178
British Columbia    107
Alberta              98
New York             84
                   ... 
United Kingdom        1
Delaware              1
Czechia               1
Oklahoma              1
RUS                   1
Name: count, Length: 69, dtype: int64


In [26]:

## Filter out any rows that don't have USA or CAN in the Country column

# print length of merged_df before filtering
print("Merged DataFrame shape before filtering for USA/CAN:", merged_df.shape)
us_can_df = merged_df[(merged_df["Country"] == "USA") | (merged_df["Country"] == "Canada")]

# print length of us_can_df after filtering
print("Filtered DataFrame shape for USA/CAN:", us_can_df.shape)

# Print summary of State_Province values in us_can_df
print("USA/CAN DataFrame State/Province Value Counts")
# USA / Canada Country Value Counts
print(us_can_df['Country'].value_counts())
# State Province Value Counts
print(us_can_df['State_Province'].value_counts())

# Check State Province values of rows filtered out
# filtered_out_df = merged_df[~merged_df.index.isin(us_can_df.index)]
# print("Filtered Out DataFrame State/Province Value Counts")
# print(filtered_out_df['State_Province'].value_counts())


Merged DataFrame shape before filtering for USA/CAN: (1464, 32)
Filtered DataFrame shape for USA/CAN: (1347, 32)
USA/CAN DataFrame State/Province Value Counts
Country
USA       810
Canada    537
Name: count, dtype: int64
State_Province
Minnesota                    184
Ontario                      178
British Columbia             107
Alberta                       98
New York                      84
Massachusetts                 72
Michigan                      69
Quebec                        66
Illinois                      54
California                    45
New Jersey                    37
Saskatchewan                  35
Pennsylvania                  30
Manitoba                      27
Connecticut                   26
Wisconsin                     26
New Hampshire                 14
Colorado                      14
Missouri                      13
North Dakota                  13
Ohio                          13
Maryland                      12
Alaska                        12
Nova 

In [27]:
### Group and aggregate stats by State_Province for USA and CAN players
state_prov_agg_stats = {
    "Games_Played": "sum",
    "G": "sum",
    "A": "sum",
    "Pts": "sum",
    "PlusMinus": "sum",
    "Shots": "sum",
    "TOI_sec": "sum",
    "PIM": "sum",
}

state_prov_agg_df = us_can_df.groupby("State_Province").agg(state_prov_agg_stats).reset_index()
print(state_prov_agg_df.head(20))

## Get count of players in each State_Province and calculate per player averages
state_prov_player_counts = (us_can_df["State_Province"]
                    .value_counts(dropna=False)
                    .fillna(0).astype(int))
print("Player Counts by State/Province")
print(state_prov_player_counts.reset_index().rename(columns={"index":"State/Province","State_Province":" Count"}))

### Divide aggregated stats by player counts to get per player averages by State_Province
state_prov_player_counts = state_prov_player_counts.reindex(state_prov_agg_df["State_Province"])
for stat in state_prov_agg_stats.keys():
    state_prov_agg_df[stat + "_per_player"] = state_prov_agg_df[stat] / state_prov_player_counts.values

## Calculate rate stats per 60 minutes for each aggregated stat by State_Province
for stat in state_prov_agg_stats.keys():
    state_prov_agg_df[stat + "_per_60min"] = (state_prov_agg_df[stat] / state_prov_agg_df["TOI_sec"]) * 3600

# Calculate per game averages for each aggregated stat by State_Province
for stat in state_prov_agg_stats.keys():
    state_prov_agg_df[stat + "_per_player_per_game_played"] = state_prov_agg_df[stat] / state_prov_agg_df["Games_Played"]
# state_prov_agg_df.info()
# state_prov_agg_df.head(20)

      State_Province  Games_Played    G    A  Pts  PlusMinus  Shots  TOI_sec  \
0             Alaska           110   19   34   53         -9    203   108615   
1            Alberta           925  140  228  368        -44   1356   827134   
2            Arizona           103   12   35   47          6    120    90964   
3   British Columbia           961  134  251  385        -23   1513   874112   
4         California           364   51   95  146          5    519   308987   
5           Colorado           107    7   15   22          5    112    90984   
6        Connecticut           207   27   48   75         16    345   183867   
7           Delaware            10    2    3    5          2     10     7951   
8            Florida           102   20   39   59          2    181    97710   
9            Georgia            59    6   14   20          7     81    44801   
10             Idaho            11    1    3    4         -1      9    16195   
11          Illinois           541   68 

## MAKE A MAP/S FOR REDACTED CHARTS ON REDDIT

In [28]:
# # Check path of shapefile

# shapefile_path = "../../data/vault/10m_state_province_file.shp"

# # try to load file
# with 


In [29]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path



def plot_state_prov_choropleth(
    state_prov_agg_df: pd.DataFrame,
    metric: str = "G",                     # or "Pts"
    shapefile_path: str = "..\\..\\data\\vault\\state_province_geo\\ne_110m_admin_1_states_provinces.shp",
    output_path: str | Path = "state_prov_map_G.png",
    title: str | None = None,
    cmap: str = "YlGnBu",                  # nice smooth gradient
    dpi: int = 300
):
    """
    Plot a static choropleth map of US states + Canadian provinces using
    state_prov_agg_df, which must have columns:
        - 'State_Province'
        - the chosen metric (e.g. 'G' or 'Pts')
    """

    # --- load the shapefile ---
    admin1 = gpd.read_file(shapefile_path)

    # Natural Earth usually has both 'name' and 'name_en'
    # After admin1 = gpd.read_file(shapefile_path)

    # See what columns you have (optional, but handy):
    print("Shapefile columns:", admin1.columns.tolist())

    # Try a few common field names
    candidate_cols = ["name_en", "name", "name_1", "NAME_1", "state_name", "STATE_NAME"]
    name_col = None
    for col in candidate_cols:
        if col in admin1.columns:
            name_col = col
            break

    if name_col is None:
        raise ValueError(
            "Could not find a suitable name column in shapefile. "
            "Available columns are: "
            + ", ".join(admin1.columns.astype(str))
        )

    admin1["region_name"] = admin1[name_col]

    

    # Keep only USA + Canada to match your table
    mask = admin1["admin"].isin(["United States of America", "Canada"])
    regions = admin1.loc[mask].copy()

    # --- join your data onto the geometry ---
    # Make sure the key column is string
    state_prov_agg_df = state_prov_agg_df.copy()
    state_prov_agg_df["State_Province"] = state_prov_agg_df["State_Province"].astype(str)

    merged = regions.merge(
        state_prov_agg_df,
        left_on="region_name",
        right_on="State_Province",
        how="left",
        validate="1:1"
    )

    if metric not in merged.columns:
        raise ValueError(f"Metric '{metric}' is not a column in the dataframe.")

    # --- plotting ---
    fig, ax = plt.subplots(figsize=(8, 10))  # tweak for aspect/Reddit

    # Blank-ish style: no axes, white background
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    # Draw boundaries lightly so it's minimalist
    merged.boundary.plot(ax=ax, linewidth=0.3, color="lightgrey")

    # Choropleth fill
    merged.plot(
        column=metric,
        ax=ax,
        cmap=cmap,
        linewidth=0.4,
        edgecolor="black",
        legend=True,
        legend_kwds={
            "label": metric,
            "shrink": 0.5,
            "orientation": "vertical"
        },
        missing_kwds={
            "color": "white",
            "edgecolor": "lightgrey",
            "hatch": "///",
            "label": "No data"
        }
    )

    # Nicely centered, mostly-blank map
    ax.set_axis_off()
    ax.set_aspect("equal")

    if title is None:
        title = f"{metric} by State / Province (College Hockey)"
    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)

    # Optional subtitle / data source
    # ax.text(0.5, 0.03, "Data: Your College Hockey DB",
    #         transform=fig.transFigure, ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    output_path = Path(output_path)
    fig.savefig(output_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    # plt.close(fig)
    # Show the plot
    plt.show()

    print(f"Map saved to: {output_path}")



## Call the function to plot goals by state/province
plot_state_prov_choropleth(
    state_prov_agg_df=state_prov_agg_df,
    metric="G",
    shapefile_path="../../data/vault/10m_state_province_file.shp",
    output_path=plot_folder / "state_prov_map_G.png",
    title="Goals by State / Province (College Hockey)",
    cmap="YlGnBu",
    dpi=300
)